# 01. Data Exploration and Cleaning
**Proyek**: Solusi Prima Packaging Study Case  
**Peran**: Data Analyst  
**Tujuan**: Memuat, memeriksa, membersihkan, dan mempersiapkan data mentah untuk analisis lanjutan.  

---

### Load Raw Data

In [1]:
import logging
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict

import pandas as pd


# =============================================================================
# KONFIGURASI LOGGING
# Menggunakan logging standar (bukan print) agar output memiliki timestamp dan
# severity level — memudahkan debugging saat pipeline dijalankan ulang.
# getLogger(__name__) memastikan log bisa difilter per modul jika notebook ini
# diimpor sebagai bagian dari pipeline yang lebih besar.
# =============================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


# =============================================================================
# KONFIGURASI PATH DAN FILE
# Semua literal path dan nama file dikumpulkan di satu tempat agar perubahan
# struktur direktori hanya perlu dilakukan sekali, bukan di setiap read_csv.
# pathlib.Path digunakan (bukan os.path.join) karena lebih ekspresif dan
# aman lintas sistem operasi (Windows/Linux/macOS).
# =============================================================================
PATH_RAW: Path = Path("..") / "data" / "raw"


@dataclass(frozen=True)
class RawFileConfig:
    """
    Kontrak antara kode dan satu file CSV raw.

    Atribut
    -------
    filename : str
        Nama file CSV di dalam PATH_RAW.
    required_columns : list
        Kolom yang wajib ada setelah load. Divalidasi sebelum pipeline
        downstream berjalan (fail-fast).
    encoding : str
        Encoding file. Default utf-8; ganti ke 'latin-1' jika file
        berasal dari Excel Indonesia yang menggunakan Windows-1252.
    dtype : Dict[str, str]
        Override tipe data kolom tertentu. Kolom ID selalu dikunci 'str'
        untuk mencegah Pandas menginterpretasikan format seperti 'CORP-01'
        secara salah.
    """
    filename: str
    required_columns: list = field(default_factory=list)
    encoding: str = "utf-8"
    dtype: Dict[str, str] = field(default_factory=dict)


# Setiap entry adalah satu kontrak dataset: nama file + kolom kritis + dtype.
# Jika skema sumber berubah (misal kolom diganti nama), cukup update di sini.
RAW_FILES: Dict[str, RawFileConfig] = {
    "transaksi": RawFileConfig(
        filename="data_transaksi.csv",
        required_columns=[
            "order_id", "tanggal", "reseller_id", "customer_id",
            "product_id", "qty", "harga", "margin",
        ],
        # Semua kolom ID dikunci str — formatnya seperti 'PO-2508-001' dan
        # 'RESL-A' tidak akan pernah valid sebagai numerik, tapi Pandas tetap
        # mencoba inferensi jika tidak dikontrol secara eksplisit.
        dtype={
            "order_id":    "str",
            "reseller_id": "str",
            "customer_id": "str",
            "product_id":  "str",
        },
    ),
    "produk": RawFileConfig(
        filename="data_produk.csv",
        required_columns=[
            "product_id", "nama_produk", "kategori", "harga_satuan", "stok",
        ],
        # harga_satuan dan stok dibiarkan tanpa override karena keduanya
        # memang integer bersih di file sumber.
        dtype={"product_id": "str"},
    ),
    "pelanggan": RawFileConfig(
        filename="data_pelanggan.csv",
        required_columns=[
            "customer_id", "nama_pelanggan", "jumlah_pembelian",
            "terakhir_beli", "feedback",
        ],
        dtype={"customer_id": "str"},
    ),
}


# =============================================================================
# FUNGSI BANTU: VALIDASI SKEMA
# =============================================================================
def _validate_schema(df: pd.DataFrame, config: RawFileConfig, label: str) -> None:
    """
    Memastikan semua kolom kritis hadir sebelum transformasi downstream.

    Validasi ini berjalan segera setelah read_csv (fail-fast), sehingga
    pipeline tidak gagal di tengah jalan karena KeyError yang sulit dilacak.

    Raises
    ------
    ValueError
        Jika satu atau lebih kolom kritis tidak ditemukan di DataFrame.
    """
    missing = set(config.required_columns) - set(df.columns)
    if missing:
        # Tampilkan daftar kolom yang hilang agar mudah diidentifikasi
        # tanpa perlu membuka file sumber secara manual.
        raise ValueError(
            f"[{label}] Schema mismatch: kolom berikut tidak ditemukan: {sorted(missing)}"
        )
    logger.info("[%s] Schema OK. Kolom kritis terpenuhi.", label)


# =============================================================================
# FUNGSI UTAMA: LOAD DATA
# =============================================================================
def load_raw_data(
    path_raw: Path = PATH_RAW,
    file_configs: Dict[str, RawFileConfig] = RAW_FILES,
) -> Dict[str, pd.DataFrame]:
    """
    Memuat semua file CSV raw dan mengembalikannya sebagai dictionary DataFrame.

    Urutan kerja tiap file:
        1. Validasi keberadaan file (FileNotFoundError jika tidak ada)
        2. Baca CSV dengan parameter eksplisit
        3. Validasi skema kolom (ValueError jika ada kolom kritis hilang)

    Parameter
    ---------
    path_raw : Path
        Direktori tempat file CSV raw berada.
    file_configs : Dict[str, RawFileConfig]
        Mapping label dataset ke konfigurasinya. Defaultnya adalah RAW_FILES
        yang sudah didefinisikan di atas; bisa di-override untuk testing.

    Returns
    -------
    Dict[str, pd.DataFrame]
        Key sesuai label di file_configs, value adalah DataFrame tervalidasi.

    Raises
    ------
    FileNotFoundError
        Jika file tidak ditemukan di path yang ditentukan.
    pd.errors.ParserError
        Jika isi file tidak bisa di-parse sebagai CSV yang valid.
    ValueError
        Jika skema DataFrame tidak memenuhi kolom kritis yang diharapkan.
    """
    dataframes: Dict[str, pd.DataFrame] = {}

    for label, config in file_configs.items():
        filepath = path_raw / config.filename
        logger.info("Memuat [%s] dari: %s", label, filepath)

        # Cek keberadaan file lebih awal (sebelum read_csv) agar pesan error
        # langsung menyebut path yang bermasalah, bukan traceback Pandas
        # yang lebih sulit dibaca oleh pembaca notebook.
        if not filepath.is_file():
            raise FileNotFoundError(
                f"[{label}] File tidak ditemukan: '{filepath}'. "
                f"Pastikan file sudah berada di direktori '{path_raw}'."
            )

        try:
            df = pd.read_csv(
                filepath,
                dtype=config.dtype if config.dtype else None,
                encoding=config.encoding,
                low_memory=False,   # paksa Pandas membaca seluruh file dalam
                                    # satu pass untuk type inference yang konsisten;
                                    # tanpa ini, kolom besar bisa mendapat tipe
                                    # yang berbeda tergantung chunk yang dibaca
            )
        except pd.errors.ParserError as exc:
            # ParserError berarti masalah pada isi file (delimiter salah,
            # quoting tidak lengkap, encoding tidak cocok) — bukan masalah path.
            # Dibedakan dari FileNotFoundError agar tindakan perbaikannya jelas.
            raise pd.errors.ParserError(
                f"[{label}] File '{filepath}' tidak dapat di-parse. "
                f"Periksa integritas CSV (delimiter, quoting, encoding). Detail: {exc}"
            ) from exc

        _validate_schema(df, config, label)

        logger.info(
            "[%s] Berhasil dimuat. Shape: %d baris x %d kolom.",
            label, df.shape[0], df.shape[1],
        )
        dataframes[label] = df

    logger.info(
        "Semua %d dataset berhasil dimuat: %s",
        len(dataframes),
        list(dataframes.keys()),
    )
    return dataframes


# =============================================================================
# EKSEKUSI (NOTEBOOK ENTRY POINT)
# Definisi fungsi dan eksekusinya dipisah agar fungsi load_raw_data() bisa
# diimpor dan diuji ulang dari notebook lain tanpa memicu side effect
# (pembacaan file otomatis) saat modul di-import.
# =============================================================================
raw_data = load_raw_data()

# Unpack ke variabel individual untuk kemudahan akses di sel-sel berikutnya
df_transaksi = raw_data["transaksi"]
df_produk    = raw_data["produk"]
df_pelanggan = raw_data["pelanggan"]

2026-06-23 05:43:03 | INFO     | __main__ | Memuat [transaksi] dari: ..\data\raw\data_transaksi.csv
2026-06-23 05:43:03 | INFO     | __main__ | [transaksi] Schema OK. Kolom kritis terpenuhi.
2026-06-23 05:43:03 | INFO     | __main__ | [transaksi] Berhasil dimuat. Shape: 100 baris x 8 kolom.
2026-06-23 05:43:03 | INFO     | __main__ | Memuat [produk] dari: ..\data\raw\data_produk.csv
2026-06-23 05:43:03 | INFO     | __main__ | [produk] Schema OK. Kolom kritis terpenuhi.
2026-06-23 05:43:03 | INFO     | __main__ | [produk] Berhasil dimuat. Shape: 8 baris x 5 kolom.
2026-06-23 05:43:03 | INFO     | __main__ | Memuat [pelanggan] dari: ..\data\raw\data_pelanggan.csv
2026-06-23 05:43:03 | INFO     | __main__ | [pelanggan] Schema OK. Kolom kritis terpenuhi.
2026-06-23 05:43:03 | INFO     | __main__ | [pelanggan] Berhasil dimuat. Shape: 12 baris x 5 kolom.
2026-06-23 05:43:03 | INFO     | __main__ | Semua 3 dataset berhasil dimuat: ['transaksi', 'produk', 'pelanggan']


### Inspeksi Data Transaksi, Produk dan Pelanggan

In [16]:
from typing import Any, List
from IPython.display import display, HTML

# =============================================================================
# KONFIGURASI PALET WARNA (MCKINSEY CORPORATE STYLE)
# =============================================================================

C: Dict[str, str] = {
    "navy": "#0B1D35",
    "tick": "#6B7B8D",
    "footnote": "#9DA8B5",
    "blue": "#1558B0",
    "forecast": "#2A7DE1",
    "slate": "rgba(94,108,132,0.38)",
    "grid": "rgba(11,29,53,0.065)",
    "axis_line": "rgba(11,29,53,0.16)",
    "bg": "#FFFFFF",
}

PALETTE: Dict[str, str] = {
    "bg_primary": "#FFFFFF",
    "bg_secondary": "#F3F5F7",
    "bg_accent": "#EAF0F8",
    "text_primary": "#0B1D35",
    "text_muted": "#9DA8B5",
    "accent_blue": "#1558B0",
    "accent_green": "#1a7f4b",
    "accent_red": "#c0392b",
    "accent_amber": "#b7791f",
    "border": "#D8DCE1",
}

# =============================================================================
# STYLESHEET CSS UNTUK JUPYTER NOTEBOOK RENDERING
# =============================================================================

BLOCK_CSS: str = f"""
<style>
.mck-wrap {{
    margin-bottom: 32px;
    background: {PALETTE['bg_primary']};
    font-family: monospace;
}}
.mck-section-header {{
    background: {PALETTE['bg_primary']};
    border: none !important;
    border-bottom: none !important;
    padding: 12px 14px;
    margin: 0 0 4px 0;
    font-family: monospace;
    font-size: 14px;
    font-weight: 700;
    color: {PALETTE['text_primary']};
    text-align: center;
}}
.mck-sublabel {{
    color: {C['navy']};
    font-family: monospace;
    font-size: 12px;
    font-weight: 600;
    margin: 0;
    padding: 10px 0 4px 0;
    background: {PALETTE['bg_primary']};
    display: block;
}}
.mck-badge-wrap {{
    margin: 8px 0 0 0;
    background: {PALETTE['bg_primary']};
    padding: 6px 0 4px 0;
}}
.mck-badge {{
    display: inline-block;
    padding: 3px 10px;
    border-radius: 4px;
    font-family: monospace;
    font-size: 12px;
    margin: 0 6px 0 0;
    border: 1px solid {PALETTE['border']};
    background: {PALETTE['bg_secondary']};
}}
.mck-wrap table {{
    border-collapse: collapse !important;
    border: 1px solid {PALETTE['border']} !important;
    background: {PALETTE['bg_primary']} !important;
    margin: 0 !important;
}}
.mck-wrap .mck-table-full table {{
    width: 100% !important;
}}
.mck-wrap .mck-table-auto table {{
    width: auto !important;
}}
.mck-wrap thead th {{
    background-color: {PALETTE['bg_secondary']} !important;
    color: {PALETTE['accent_blue']} !important;
    font-family: monospace !important;
    font-size: 12px !important;
    font-weight: 700 !important;
    border-bottom: 2px solid {PALETTE['accent_blue']} !important;
    padding: 8px 12px !important;
    text-align: left !important;
    letter-spacing: 0.03em !important;
}}
.mck-wrap tbody th {{
    background-color: {PALETTE['bg_secondary']} !important;
    color: {C['tick']} !important;
    font-family: monospace !important;
    font-size: 11px !important;
    border-right: 1px solid {PALETTE['border']} !important;
    padding: 6px 10px !important;
    text-align: left !important;
}}
.mck-wrap tbody td {{
    background-color: {PALETTE['bg_primary']} !important;
    color: {PALETTE['text_primary']} !important;
    font-family: monospace !important;
    font-size: 12px !important;
    border-bottom: 1px solid {PALETTE['border']} !important;
    padding: 6px 12px !important;
    text-align: left !important;
}}
.mck-wrap caption {{
    caption-side: top !important;
    text-align: left !important;
    padding: 0 0 4px 0 !important;
    color: {C['footnote']} !important;
    font-size: 11px !important;
    font-family: monospace !important;
}}
</style>
"""

BASE_TABLE_CSS: List[Dict[str, Any]] = [
    {"selector": "thead th", "props": [
        ("background-color", PALETTE["bg_secondary"]),
        ("color",            PALETTE["accent_blue"]),
        ("font-family",      "monospace"),
        ("font-size",        "12px"),
        ("font-weight",      "700"),
        ("border-bottom",    f"2px solid {PALETTE['accent_blue']}"),
        ("padding",          "8px 12px"),
        ("text-align",       "left"),
    ]},
    {"selector": "tbody th", "props": [
        ("background-color", PALETTE["bg_secondary"]),
        ("color",            C["tick"]),
        ("font-family",      "monospace"),
        ("font-size",        "11px"),
        ("border-right",     f"1px solid {PALETTE['border']}"),
        ("padding",          "6px 10px"),
        ("text-align",       "left"),
    ]},
    {"selector": "tbody td", "props": [
        ("background-color", PALETTE["bg_primary"]),
        ("color",            PALETTE["text_primary"]),
        ("font-family",      "monospace"),
        ("font-size",        "12px"),
        ("border-bottom",    f"1px solid {PALETTE['border']}"),
        ("padding",          "6px 12px"),
        ("text-align",       "left"),
    ]},
    {"selector": "table", "props": [
        ("border-collapse", "collapse"),
        ("border",          f"1px solid {PALETTE['border']}"),
        ("background",      PALETTE["bg_primary"]),
        ("margin",          "0"),
    ]},
    {"selector": "caption", "props": [
        ("caption-side", "top"),
        ("text-align",   "left"),
        ("padding",      "0 0 4px 0"),
    ]},
]

# =============================================================================
# FUNGSI INTERNAL (MANIPULASI & STYLING HTML)
# =============================================================================

def _style_head(df: pd.DataFrame, n: int = 5) -> str:
    """Mengonversi n baris pertama DataFrame menjadi tabel HTML dengan gaya zebra-striping.

    Args:
        df (pd.DataFrame): DataFrame sumber yang akan diambil sampel datanya.
        n (int, optional): Jumlah baris yang ingin ditampilkan dari atas. Nilai default adalah 5.

    Returns:
        str: String berformat HTML yang membungkus tabel sampel di dalam container penuh (`mck-table-full`).
    """
    bg_accent = PALETTE["bg_accent"]
    bg_primary = PALETTE["bg_primary"]
    text_primary = PALETTE["text_primary"]

    def zebra(row: pd.Series) -> List[str]:
        """Fungsi internal pemetaan warna baris berdasarkan indeks (genap/ganjil)."""
        bg = bg_accent if getattr(row, "name", 0) % 2 == 0 else bg_primary
        style_str = f"background-color: {bg} !important; color: {text_primary} !important; text-align: left !important"
        return [style_str for _ in row]

    table_html = (
        df.head(n)
        .style
        .apply(zebra, axis=1)
        .set_table_styles(BASE_TABLE_CSS)
        .to_html()
    )
    return f"<div class='mck-table-full'>{table_html}</div>"


def _build_schema_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Mengekstrak meta-data skema dan metrik kelengkapan nilai dari DataFrame.

    Args:
        df (pd.DataFrame): DataFrame target analisis skema.

    Returns:
        pd.DataFrame: DataFrame baru berisi ringkasan kolom, tipe data, jumlah non-null,
            jumlah null, dan persentase missing value.
    """
    null_pct_raw = (df.isnull().mean() * 100).values
    summary = pd.DataFrame({
        "Kolom":       df.columns,
        "Tipe Data":   [str(dt) for dt in df.dtypes],
        "Non-Null":    df.notnull().sum().values,
        "Null Count": df.isnull().sum().values,
        "Null %":     [f"{v:.2f}" for v in null_pct_raw],
    })
    return summary.reset_index(drop=True)


def _style_schema(schema_df: pd.DataFrame) -> str:
    """Menerapkan pewarnaan indikator kualitas data (*conditional formatting*) pada tabel skema.

    Setiap kolom kritikal seperti jumlah missing value atau persentase null akan diberi
    warna peringatan jika ditemukan anomali atau data kosong (> 0).

    Args:
        schema_df (pd.DataFrame): DataFrame hasil dari fungsi `_build_schema_summary`.

    Returns:
        str: String HTML tabel skema yang telah diformat dan dibungkus container `mck-table-auto`.
    """
    color_red = PALETTE["accent_red"]
    color_green = PALETTE["accent_green"]
    color_amber = PALETTE["accent_amber"]
    color_forecast = C["forecast"]
    color_primary = PALETTE["text_primary"]
    bg_primary = PALETTE["bg_primary"]

    def bg_all(row: pd.Series) -> List[str]:
        """Mengunci warna dasar seluruh sel agar seragam lintas tema Jupyter."""
        style_str = f"background-color: {bg_primary} !important; color: {color_primary} !important; text-align: left !important"
        return [style_str for _ in row]

    def color_null_count(val: Any) -> str:
        """Memberikan penanda tebal merah jika terdapat missing value."""
        base = "text-align: left !important; font-family: monospace"
        if isinstance(val, (int, float)) and val > 0:
            return f"color: {color_red} !important; font-weight: 700; {base}"
        if isinstance(val, (int, float)) and val == 0:
            return f"color: {color_green} !important; {base}"
        return f"color: {color_primary} !important; {base}"

    def color_null_pct(val: Any) -> str:
        """Memberikan penanda tebal merah pada persentase missing value jika > 0%."""
        base = "text-align: left !important; font-family: monospace"
        try:
            numeric = float(val)
            if numeric > 0:
                return f"color: {color_red} !important; font-weight: 700; {base}"
            return f"color: {color_green} !important; {base}"
        except (ValueError, TypeError):
            return f"color: {color_primary} !important; {base}"

    def color_dtype(val: Any) -> str:
        """Memformat estetika teks penulisan tipe data."""
        return f"color: {color_amber} !important; font-family: monospace; font-size: 11px; text-align: left !important"

    def color_nonnull(val: Any) -> str:
        """Memformat estetika teks penulisan jumlah data non-null."""
        return f"color: {color_forecast} !important; font-family: monospace; text-align: left !important"

    table_html = (
        schema_df.style
        .apply(bg_all, axis=1)
        .map(color_null_count, subset=["Null Count"])
        .map(color_null_pct,   subset=["Null %"])
        .map(color_dtype,      subset=["Tipe Data"])
        .map(color_nonnull,    subset=["Non-Null"])
        .set_table_styles(BASE_TABLE_CSS)
        .hide(axis="index")
        .to_html()
    )
    return f"<div class='mck-table-auto'>{table_html}</div>"


def _build_quality_badge(df: pd.DataFrame) -> str:
    """Menghitung metrik integritas tingkat tinggi (duplikasi & null global) menjadi elemen HTML badge.

    Fungsi ini melakukan pengecekan cepat (fail-fast indicator) untuk mendeteksi baris duplikat
    atau total data kosong di seluruh sel DataFrame.

    Args:
        df (pd.DataFrame): DataFrame yang akan dihitung metrik kualitasnya.

    Returns:
        str: Blok HTML berisi barisan komponen *badge* dengan ikon penanda status (✓ atau ✗).
    """
    n_dup = int(df.duplicated().sum())
    n_null = int(df.isnull().sum().sum())

    dup_color = PALETTE["accent_red"] if n_dup > 0 else PALETTE["accent_green"]
    null_color = PALETTE["accent_red"] if n_null > 0 else PALETTE["accent_green"]
    dim_color = C["navy"]

    dup_icon = "✗" if n_dup > 0 else "✓"
    null_icon = "✗" if n_null > 0 else "✓"

    return (
        f"<div class='mck-badge-wrap'>"
        f"<span class='mck-badge' style='color:{dup_color}'>"
        f"{dup_icon} Duplikat: {n_dup}</span>"
        f"<span class='mck-badge' style='color:{null_color}'>"
        f"{null_icon} Total Null: {n_null}</span>"
        f"<span class='mck-badge' style='color:{dim_color}'>"
        f"⊞ {df.shape[0]} baris × {df.shape[1]} kolom</span>"
        f"</div>"
    )

# =============================================================================
# FUNGSI ANTARMUKA UTAMA (PUBLIC API)
# =============================================================================

def inspect_dataframes(datasets: Dict[str, pd.DataFrame], n_head: int = 5) -> None:
    """Melakukan inspeksi komprehensif terhadap sekumpulan DataFrame dan merendernya ke Jupyter DOM.

    Fungsi ini menyatukan visualisasi sampel data, analisis metadata skema, dan metrik kualitas
    ke dalam satu kesatuan arsitektur HTML per dataset. Setiap eksekusi menginjeksikan aturan CSS lokal
    untuk memastikan rendering tidak merusak komponen global notebook.

    Args:
        datasets (Dict[str, pd.DataFrame]): Mapping atau kamus yang berisi nama label (key) 
            dan objek DataFrame (value) yang akan diinspeksi.
        n_head (int, optional): Jumlah baris sampel data teratas yang ditampilkan. Nilai default adalah 5.
    """
    for label, df in datasets.items():
        head_html = _style_head(df, n=n_head)
        schema_html = _style_schema(_build_schema_summary(df))
        badge_html = _build_quality_badge(df)

        block = (
            f"{BLOCK_CSS}"
            f"<div class='mck-wrap'>"
            f"<div class='mck-section-header'>df_{label}</div>"
            f"<span class='mck-sublabel'>Sample Data</span>"
            f"{head_html}"
            f"<span class='mck-sublabel' style='padding-top:16px'>Schema &amp; Kelengkapan</span>"
            f"{schema_html}"
            f"{badge_html}"
            f"</div>"
        )
        display(HTML(block))

# =============================================================================
# ENTRY POINT EKSEKUSI
# =============================================================================

inspect_dataframes({
    "transaksi": df_transaksi,
    "produk":    df_produk,
    "pelanggan": df_pelanggan,
})

### Proses Pembersihan Data

In [18]:
# =============================================================================
# FUNGSI PEMBERSIHAN DATA
# =============================================================================

def clean_datetime_columns(datasets: Dict[str, pd.DataFrame], datetime_configs: Dict[str, List[str]]) -> Dict[str, pd.DataFrame]:
    """Mengonversi kolom target pada dataset tertentu menjadi tipe data datetime.

    Menggunakan pendekatan fail-fast untuk memastikan kolom yang ditargetkan 
    tersedia di dalam DataFrame sebelum melakukan konversi. Menggunakan assignment
    langsung untuk menghindari kegagalan tipe data jika kolom sebelumnya dikunci sebagai string.

    Args:
        datasets (Dict[str, pd.DataFrame]): Kamus berisi nama label dan objek DataFrame.
        datetime_configs (Dict[str, List[str]]): Pemetaan nama label dataset ke daftar 
            nama kolom yang ingin dikonversi menjadi datetime.

    Returns:
        Dict[str, pd.DataFrame]: Kamus dataset dengan kolom datetime yang sudah diperbarui.

    Raises:
        KeyError: Jika dataset atau kolom yang dikonfigurasi tidak ditemukan.
    """
    cleaned_datasets = datasets.copy()
    
    for label, columns in datetime_configs.items():
        if label not in cleaned_datasets:
            raise KeyError(f"Dataset dengan nama label '{label}' tidak ditemukan dalam input.")
            
        df = cleaned_datasets[label]
        
        for col in columns:
            if col not in df.columns:
                raise KeyError(f"Kolom '{col}' tidak ditemukan pada dataset '{label}'.")
            
            logger.info("Mengonversi kolom '%s' pada dataset '%s' ke datetime.", col, label)
            
            # Menggunakan assignment langsung untuk memaksa Pandas mengubah skema tipe data kolom
            cleaned_datasets[label][col] = pd.to_datetime(df[col])
            
    return cleaned_datasets


def strip_whitespace_inplace(datasets: Dict[str, pd.DataFrame]) -> None:
    """Menghapus whitespace di awal dan akhir string untuk semua kolom berbasis teks.

    Fungsi ini melakukan modifikasi langsung pada objek DataFrame yang diberikan 
    (inplace) untuk menghemat alokasi memori pada pipeline data besar.

    Args:
        datasets (Dict[str, pd.DataFrame]): Kamus berisi nama label dan objek DataFrame.
    """
    for label, df in datasets.items():
        text_cols = df.select_dtypes(include=["object", "string"]).columns
        if not text_cols.empty:
            logger.info("Membersihkan whitespace untuk kolom teks pada dataset '%s': %s", label, list(text_cols))
            for col in text_cols:
                datasets[label].loc[:, col] = df[col].astype(str).str.strip()


# =============================================================================
# EKSEKUSI PIPELINE
# =============================================================================

def run_data_cleaning_pipeline(datasets: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """Eksekutor utama untuk seluruh rangkaian proses pembersihan data.

    Args:
        datasets (Dict[str, pd.DataFrame]): Kamus berisikan df_transaksi, df_produk, 
            dan df_pelanggan.

    Returns:
        Dict[str, pd.DataFrame]: Kamus berisi DataFrame yang telah dibersihkan secara total.
    """
    logger.info("=== MEMULAI PROSES PEMBERSIHAN DATA ===")
    
    # 1. Konfigurasi kolom datetime terpusat
    datetime_configs = {
        "transaksi": ["tanggal"],
        "pelanggan": ["terakhir_beli"]
    }
    
    # 2. Eksekusi konversi datetime
    processed_datasets = clean_datetime_columns(datasets, datetime_configs)
    
    # 3. Eksekusi pembersihan whitespace
    strip_whitespace_inplace(processed_datasets)
    
    logger.info("Proses pembersihan selesai. Tipe data tanggal dan teks sudah diperbarui.")
    return processed_datasets


cleaned_data = run_data_cleaning_pipeline({
    "transaksi": df_transaksi,
    "produk":    df_produk,
    "pelanggan": df_pelanggan,
})

2026-06-23 06:57:53 | INFO     | __main__ | === MEMULAI PROSES PEMBERSIHAN DATA ===
2026-06-23 06:57:53 | INFO     | __main__ | Mengonversi kolom 'tanggal' pada dataset 'transaksi' ke datetime.
2026-06-23 06:57:53 | INFO     | __main__ | Mengonversi kolom 'terakhir_beli' pada dataset 'pelanggan' ke datetime.
2026-06-23 06:57:53 | INFO     | __main__ | Membersihkan whitespace untuk kolom teks pada dataset 'transaksi': ['order_id', 'reseller_id', 'customer_id', 'product_id']
2026-06-23 06:57:53 | INFO     | __main__ | Membersihkan whitespace untuk kolom teks pada dataset 'produk': ['product_id', 'nama_produk', 'kategori']
2026-06-23 06:57:53 | INFO     | __main__ | Membersihkan whitespace untuk kolom teks pada dataset 'pelanggan': ['customer_id', 'nama_pelanggan', 'feedback']
2026-06-23 06:57:53 | INFO     | __main__ | Proses pembersihan selesai. Tipe data tanggal dan teks sudah diperbarui.


### Menyimpan Data ke Folder Processed

In [19]:
# =============================================================================
# KONFIGURASI PATH
# =============================================================================
DEFAULT_PATH_PROCESSED: Path = Path("..") / "data" / "processed"


# =============================================================================
# FUNGSI EKSPOR DATA
# =============================================================================

def export_cleaned_datasets(datasets: Dict[str, pd.DataFrame], output_dir: Path = DEFAULT_PATH_PROCESSED) -> None:
    """Menyimpan seluruh DataFrame yang telah dibersihkan ke dalam format CSV.

    Fungsi ini otomatis membuat direktori tujuan jika belum tersedia di dalam sistem,
    serta menangani galat operasional terkait I/O penyimpanan file.

    Args:
        datasets (Dict[str, pd.DataFrame]): Kamus berisi label nama file sebagai key
            dan objek DataFrame bersih sebagai value.
        output_dir (Path, optional): Jalur direktori penyimpanan berbasis pathlib.Path.
            Nilai default diarahkan ke DEFAULT_PATH_PROCESSED.

    Raises:
        OSError: Jika sistem operasi gagal membuat direktori atau menulis file ke disk.
    """
    logger.info("=== MEMULAI PROSES PENYIMPANAN DATA BERSIH ===")
    
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        logger.error("Gagal membuat direktori tujuan di '%s'. Detail: %s", output_dir, exc)
        raise

    for label, df in datasets.items():
        filename = f"cleaned_data_{label}.csv"
        filepath = output_dir / filename
        
        logger.info("Menyimpan dataset [%s] ke: %s", label, filepath)
        
        try:
            df.to_csv(filepath, index=False)
        except OSError as exc:
            logger.error("Gagal menulis file '%s' ke disk. Periksa izin akses atau kapasitas penyimpanan. Detail: %s", filepath, exc)
            raise

    logger.info("Semua data bersih berhasil disimpan secara aman di folder: '%s'", output_dir)


# =============================================================================
# EKSEKUSI (CELL NOTEBOOK)
# =============================================================================

export_cleaned_datasets({
    "transaksi": df_transaksi,
    "produk":    df_produk,
    "pelanggan": df_pelanggan,
})

2026-06-23 07:00:28 | INFO     | __main__ | === MEMULAI PROSES PENYIMPANAN DATA BERSIH ===
2026-06-23 07:00:28 | INFO     | __main__ | Menyimpan dataset [transaksi] ke: ..\data\processed\cleaned_data_transaksi.csv
2026-06-23 07:00:28 | INFO     | __main__ | Menyimpan dataset [produk] ke: ..\data\processed\cleaned_data_produk.csv
2026-06-23 07:00:28 | INFO     | __main__ | Menyimpan dataset [pelanggan] ke: ..\data\processed\cleaned_data_pelanggan.csv
2026-06-23 07:00:28 | INFO     | __main__ | Semua data bersih berhasil disimpan secara aman di folder: '..\data\processed'
